In [12]:
import os
import json
from tqdm import tqdm
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_chroma import Chroma
import chromadb

In [13]:
from openai import OpenAI

NIM_API_KEY = os.getenv("NVIDIA_API_KEY")

client = OpenAI(
    api_key=NIM_API_KEY,
    base_url="https://integrate.api.nvidia.com/v1"
)

CHAT_MODEL = "meta/llama-3.3-70b-instruct"

In [14]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)


In [17]:
pdf_loader = PyPDFDirectoryLoader("tesla-annual-reports")
docs = pdf_loader.load()

In [16]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
encoding_name='cl100k_base',
    chunk_size=900,
    chunk_overlap=100
)
chunks = text_splitter.split_documents(docs)

for i, chunk in enumerate(chunks):
    source = os.path.basename(chunk.metadata.get("source","unknown"))

    year = "".join([c for c in source if c.isdigit()])[:4]

    chunk.metadata.update({
        "chunk_id": f"tsla_chunk_{i}",
        "source_doc": source,
        "fiscal_year": year
    })

print("Chunks:", len(chunks))

Chunks: 2180


In [18]:
client_db = chromadb.PersistentClient(path="./tesla_db")

baseline_store = Chroma(
    collection_name="tesla_baseline",
    embedding_function=embedding_model,
    client=client_db
)
for i in tqdm(range(0, len(chunks), 200)):
    baseline_store.add_documents(
        chunks[i:i+200]
    )

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
  0%|          | 0/11 [00:51<?, ?it/s]


KeyboardInterrupt: 

In [19]:
SYSTEM_PROMPT = '''
You are a financial analyst.

Generate exactly 3 hypothetical questions answerable using ONLY the supplied chunk.

Rules:
- Questions must be grounded in the chunk.
- Include factual, analytical, risk-oriented and business questions.
- Do not invent facts.
- Return one question per line.
'''

In [20]:
import json
import os

CACHE_FILE = "hq_cache.json"

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "r", encoding="utf-8") as f:
        hq_cache = json.load(f)
else:
    hq_cache = {}

def generate_questions(chunk_text):

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0.2,
        max_tokens=500,
        messages=[
            {
                "role": "system",
                "content": """
Generate exactly 3 hypothetical questions that can be answered using ONLY the provided chunk.

Rules:
- Questions must be grounded in the chunk.
- No hallucinations.
- Questions should improve retrieval quality.
- Return ONLY:

Q1: ...
Q2: ...
Q3: ...

Do not return anything else.
"""
            },
            {
                "role": "user",
                "content": chunk_text
            }
        ]
    )

    raw = response.choices[0].message.content

    questions = []

    for line in raw.split("\n"):
        line = line.strip()

        if line.startswith("Q1:"):
            questions.append(line.replace("Q1:", "").strip())

        elif line.startswith("Q2:"):
            questions.append(line.replace("Q2:", "").strip())

        elif line.startswith("Q3:"):
            questions.append(line.replace("Q3:", "").strip())

    return questions

In [ ]:
import time
import json

chunk_lookup = {
    chunk.metadata["chunk_id"]: chunk
    for chunk in chunks
}

remaining_chunks = [
    chunk
    for chunk in chunks
    if chunk.metadata["chunk_id"] not in hq_cache
]

print("Remaining chunks:", len(remaining_chunks))
MAX_RETRIES = 10

for chunk in tqdm(remaining_chunks):
    chunk_id = chunk.metadata["chunk_id"]

    success = False
    for attempt in range(MAX_RETRIES):

        try:
            questions = generate_questions( chunk.page_content)
            hq_cache[chunk_id] = questions

            with open(
                CACHE_FILE,
                "w",
                encoding="utf-8"
            ) as f:

                json.dump( hq_cache,f,indent=2,)

            success = True

            time.sleep(2)

            break

        except Exception as e:

            if "429" in str(e):
                wait_time = min(60,5 * (attempt + 1))
                time.sleep(wait_time)
                break

    if not success:

        print(f"\nSkipped {chunk_id} after {MAX_RETRIES} retries")

hypothetical_documents = []
for chunk_id, questions in hq_cache.items():

    parent = chunk_lookup[chunk_id]
    for q in questions:
        hypothetical_documents.append(
            Document(
                page_content=q,
                metadata={
                    "parent_chunk_id": chunk_id,
                    "source_doc": parent.metadata["source_doc"],
                    "fiscal_year": parent.metadata["fiscal_year"]
                }
            )
        )

print("\nGenerated HQ docs:",
    len(hypothetical_documents))

print("Cached chunks:",
    len(hq_cache))

Remaining chunks: 2146


 47%|████▋     | 1003/2146 [4:37:28<5:27:24, 17.19s/it] 

In [ ]:
hq_store = Chroma(
    collection_name="tesla_hq",
    embedding_function=embedding_model,
    client=client_db
)

from tqdm import tqdm

BATCH_SIZE = 500

for i in tqdm(
    range(0, len(hypothetical_documents), BATCH_SIZE)
):
    hq_store.add_documents(
        hypothetical_documents[i:i+BATCH_SIZE]
    )

print("HQ vector store created successfully")

In [ ]:
def baseline_retrieve(query,k=5):
    return baseline_store.similarity_search(query,k=k)


chunk_lookup = {
    c.metadata["chunk_id"]: c
    for c in chunks
}

def hq_retrieve(query,k=8):

    hq_hits = hq_store.similarity_search(query,k=k)

    parent_ids = list({
        hit.metadata["parent_chunk_id"]
        for hit in hq_hits
    })

    parent_docs = [
        chunk_lookup[pid]
        for pid in parent_ids
        if pid in chunk_lookup
    ]

    return hq_hits,parent_docs

In [ ]:
def answer_question(query,retrieved_docs):

    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )

    prompt = f'''
Question:
{query}

Context:
{context}

Answer only from context.
Include citations using:
[source_doc | chunk_id]
'''

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[
            {"role":"user","content":prompt}
        ]
    )

    return response.choices[0].message.content

In [ ]:
benchmark_questions = {
    "HQ1":"What should a board member ask about risks that could prevent Tesla from meeting production, delivery, or scaling expectations?",
    "HQ2":"How should an analyst investigate the relationship between product defects, warranty or service obligations, customer trust, and brand risk?",
    "HQ3":"What evidence helps determine whether future cash flow depends more on capital expenditure discipline, working capital, or operating income?",
    "HQ4":"Which disclosures help evaluate technology, cybersecurity, data, or AI operational risk even if the user does not explicitly say cybersecurity?"
}

In [ ]:
results = []

for qid, query in benchmark_questions.items():

    baseline_docs = baseline_retrieve(query)

    hq_hits, parent_docs = hq_retrieve(query)

    baseline_answer = answer_question(query, baseline_docs)
    hq_answer = answer_question(query, parent_docs)

    result = {
        "question_id": qid,
        "user_query": query,
        "retrieved_hypothetical_questions":[
            {
                "hypothetical_question": h.page_content,
                "parent_chunk_id": h.metadata["parent_chunk_id"],
                "source_doc": h.metadata["source_doc"]
            }
            for h in hq_hits
        ],
        "parent_chunks_used":[
            {
                "chunk_id": d.metadata["chunk_id"],
                "source_doc": d.metadata["source_doc"]
            }
            for d in parent_docs
        ],
        "baseline_answer": baseline_answer,
        "final_answer": hq_answer
    }

    results.append(result)

In [ ]:
with open("assignment3_results.json","w",encoding="utf-8") as f:
    json.dump(results,f,indent=2)

print("Saved assignment3_results.json")

In [ ]:
assignment3_results = json.loads(
    open("assignment3_results.json","r",encoding="utf-8").read()
)
for res in assignment3_results:
    print(f"\n\nQuestion ID: {res['question_id']}")
    print(f"User Query: {res['user_query']}")
    print("\nBaseline Answer:")
    print(res["baseline_answer"])
    print("\nHQ-Enhanced Answer:")
    print(res["final_answer"])
    break